# Shin et al. (2025) from dandiset 000248: the Neuropixels claims

**Paper.** Shin H, Ogando MB, Abdeladim L, ... Lecoq J, Adesnik H (2025). Recurrent pattern
completion drives the neocortical representation of sensory inference. *Nature Neuroscience* 28,
2319-2329. [doi:10.1038/s41593-025-02055-5](https://doi.org/10.1038/s41593-025-02055-5). The
claims file was written from the version of record (main text, figure legends, Methods and
Extended Data legends); the Supplementary Information was not available to it.

**Dandiset.** [DANDI:000248](https://dandiarchive.org/dandiset/000248/0.240502.2344), version
0.240502.2344. The paper cites the dandiset without a version. This is its only published version,
so it is pinned. The draft has the same 78 assets and size; it was last modified at the time of
publication (2024-05-02). The paper's analysis code reads a copy named `00248_v240130` (January
2024), before this version; no earlier version was published, so a difference between the two
cannot be checked.

**What the dandiset holds.** The 12 Neuropixels sessions (12 mice), each a `_ogen.nwb` file with
the units of all six probes, the stimulus tables, the receptive field and size blocks, and in 10
sessions eye tracking; and 66 `_probe-N_ecephys.nwb` files with LFP, which are not used. The 2p
imaging, 2p mesoscope and 2p holography data of the paper are not in it.

**Method sources.** The claims file gives each method; where it leaves a detail open, the detail
is taken from the paper or from the analysis code the claims file cites
(github.com/hs13/IC_paper_code at commit 50e04ea), and the helper module `illusion.py` says which.

| Claims | Status |
|---|---|
| 18 Neuropixels claims (unit counts, functional subsets, receptive fields, fixed gaze, eye, cross-correlograms, decoding) | Written here; blocked (too_large): executed only after the author's time limit, outputs not reviewed |
| supp3-decoder-types | Blocked (too_large): the network of the Methods is estimated at hours |
| 21 claims on the 2p imaging, mesoscope and holography data | Blocked (data_missing): not in the dandiset |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import warnings
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats

import illusion as il

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
N_JOBS = 4

started = time.time()
sessions = il.load_all()
print(f"loaded {len(sessions)} sessions of dandiset {il.DANDISET_ID} version {il.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
assets = pd.DataFrame(il.all_assets())
assets["subject"] = assets.path.str.split("/").str[0]
assets["kind"] = np.where(assets.path.str.endswith("_ogen.nwb"), "session (_ogen)", "probe LFP (_ecephys)")
print("Assets:", len(assets), "; total size (GB):", round(assets["size"].sum() / 1e9, 1))
print(pd.crosstab(assets.subject, assets.kind).to_string())
info = pd.DataFrame([{"subject": s["subject"], "session": s["session_id"], "genotype": s["genotype"].split("/")[0],
                      "age": s["age"], "units_in_file": s["n_units_file"],
                      "visual_area_units": len(s["units"]), "eye_tracking": s["eye"] is not None}
                     for s in sessions])
print()
print(info.to_string(index=False))
print("\nMice by genotype:", info.genotype.value_counts().to_dict())
print("Paper, Methods (Animals): Sst-IRES-Cre;Ai32 n = 10 and PV-IRES-Cre;Ai32 n = 4; Results and legends: 12 sessions from 12 mice.")

The dandiset holds 12 sessions from 12 mice, as the Results and legends state. One file per session
(`_ogen.nwb`) holds the units of all probes; sub-619293 has no probe LFP files. The genotypes are 9
Sst-IRES-Cre;Ai32 and 3 Pvalb-IRES-Cre;Ai32 mice, against the 10 and 4 (14 mice) of the Methods'
Animals list: the Methods list two more mice than the analyses use, and the split differs.

In [ ]:
rows = []
for s in sessions:
    wd, q, ar = s["waveform_duration_all"], s["quality_all"], s["area_all"]
    rows.append({"subject": s["subject"], "units": len(wd), "noise_label": int(np.sum(q == "noise")),
                 "in_6_visual_areas": int(np.sum(ar != None)),
                 "RS_all_areas": int(np.sum(wd > 0.4)),
                 **{f"{a}_RS": int(np.sum((s["units"].area == a) & s["units"].rs)) for a in il.AREAS}})
units_tab = pd.DataFrame(rows)
print(units_tab.to_string(index=False))
allwd = np.concatenate([s["waveform_duration_all"] for s in sessions])
print("\nwaveform_duration (ms): min %.3f, median %.3f, max %.3f; fraction > 0.4 ms: %.3f" %
      (allwd.min(), np.median(allwd), allwd.max(), np.mean(allwd > 0.4)))
print("quality values:", sorted(set(np.concatenate([s["quality_all"] for s in sessions]))))
labs = pd.concat([s["unit_labels_all"] for s in sessions]).groupby(level=0).sum()
vis = labs[[il.area_of(x) is not None for x in labs.index]]
print("\nVisual cortex labels used (units):", vis.to_dict())
print("Other labels with 'VIS':", [x for x in labs.index if "VIS" in x and il.area_of(x) is None])
print("Spike times sorted within units (first 50 units per session):", all(s["sorted_ok"] for s in sessions))

Units carry a waveform duration in ms (0.03 to 2.7 ms, a bimodal range as expected for
trough-to-peak widths), a peak channel whose CCF label gives the area and layer, and a `quality`
label of `good` or `noise`. The Methods say that "all units not classified as noise were packaged
into Neurodata Without Borders files"; the files nevertheless hold units labelled `noise` (counts
above). The analyses below keep them, as the paper's code does (it applies no quality filter), and
the counts in the claims below show the paper's numbers include them. LM has fewer than 50 RS units
in four sessions (none in three), and RL none in one, as the Fig. 3d legend's session counts imply.

In [ ]:
rows = []
for s in sessions:
    for blk, b in s["blocks"].items():
        codes, n = np.unique(b["codes"], return_counts=True)
        rows.append({"subject": s["subject"], "block": blk, "table_rows": b["n_rows"], "trials": len(b["codes"]),
                     "ITI_rows_with_nonzero_frame": b["iti_rows_nonzero_frame"],
                     "n_IC1": int(np.sum(b["codes"] == il.IC1)), "n_other_min": int(n.min()), "n_other_max": int(n.max()),
                     "dur_s_min": b["duration"].min().round(3), "dur_s_max": b["duration"].max().round(3),
                     "iti_s_median": np.median(b["iti"]).round(3)})
blk_tab = pd.DataFrame(rows)
print(blk_tab.groupby("block").agg({"trials": ["min", "max"], "table_rows": ["min", "max"], "ITI_rows_with_nonzero_frame": "max",
                                    "n_IC1": ["min", "max"], "n_other_min": "min", "n_other_max": "max",
                                    "dur_s_min": "min", "dur_s_max": "max", "iti_s_median": ["min", "max"]}).to_string())
b = sessions[0]["blocks"]["ICwcfg1"]
codes, n = np.unique(b["codes"], return_counts=True)
print("\nICwcfg1 trials per image code, first session:", dict(zip(codes.tolist(), n.tolist())))
rf = pd.DataFrame([{"subject": s["subject"], "rf_rows": s["rf"]["n_rows"], "rf_trials": len(s["rf"]["pos"]),
                    "classic_per_position": np.bincount(s["rf"]["pos"][~s["rf"]["inverse"]], minlength=9).tolist(),
                    "inverse_per_position": np.bincount(s["rf"]["pos"][s["rf"]["inverse"]], minlength=9).tolist(),
                    "row_dur_s": (round(s["rf"]["row_dur"].min(), 3), round(s["rf"]["row_dur"].max(), 3)),
                    "directions_per_trial": s["rf"]["directions_per_trial"][0],
                    "longest_spontaneous_s": round(s["spont_interval"][1] - s["spont_interval"][0], 1)}
                   for s in sessions])
print(rf.to_string(index=False))

Each IC block table alternates image rows with ITI rows whose `frame` is 0, as the code assumes; no
ITI row carries an image. ICwcfg1 has 5,300 trials per session: 400 of blank, X, TC1, IC1, LC1,
TC2, LC2, IC2, TRE1, TRE2, XRE1 and XRE2, and 50 of IRE1, IRE2 and the eight segment images. The
Methods say IC and LC images were shown 400 times and each other image at least 50 times; the
data agree, with TRE, XRE, X and TC images at 400. The other blocks have 30 trials of each of the
22 images. Images last 0.400 s with a 0.4 s ITI, as stated. The receptive field block has 180
trials (four 0.25 s directions each, 0 s ITI) at 9 positions for standard and inverse patches,
10 each, as the Methods state. The stimulus tables carry the image index (`frame`) without image
names; the names are the code's list, and the image templates stored in the files are consistent
with it (shown at the end of this section).

In [ ]:
# Clock check: decode the four training images from V1 in the 400 ms after onset, in the 400 ms
# before onset, and after shuffling the labels. If spike times and the trial table share a clock,
# only the first is above chance.
def clock_check(i, subject, b, m):
    rng = np.random.default_rng(100 + i)
    shuffled = b["codes"].copy()
    sel = np.isin(shuffled, [il.IC1, il.LC1, il.LC2, il.IC2])
    shuffled[sel] = rng.permutation(shuffled[sel])
    return {"subject": subject,
            "post_0_400ms": il.decode(b["R"][:, m], b["codes"], [106, 107, 110, 111], [1105], seed=i)["acc"],
            "pre_-400_0ms": il.decode(b["R_pre"][:, m], b["codes"], [106, 107, 110, 111], [1105], seed=i)["acc"],
            "post_shuffled_labels": il.decode(b["R"][:, m], shuffled, [106, 107, 110, 111], [1105], seed=i)["acc"],
            "rate_post_Hz": b["R"][:, m].mean(), "rate_pre_Hz": b["R_pre"][:, m].mean()}
def v1_block(s):
    m = ((s["units"].area == "V1") & s["units"].rs).values
    b = s["blocks"]["ICwcfg1"]
    return {"R": b["R"][:, m], "R_pre": b["R_pre"][:, m], "codes": b["codes"]}, np.ones(m.sum(), bool)
clock = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(clock_check)(i, s["subject"], *v1_block(s)) for i, s in enumerate(sessions)))
print(clock.round(3).to_string(index=False))
print("chance = 0.25")

In every session the four training images are decoded from V1 spikes after onset and not from the
400 ms before onset or with shuffled labels (values above), so the spike times and the stimulus
table share a clock. The eye tracking timestamps are checked below.

In [ ]:
rows = []
for s in sessions:
    e = s["eye"]
    if e is None:
        rows.append({"subject": s["subject"], "eye": False})
        continue
    rows.append({"subject": s["subject"], "eye": True, "frame_rate_Hz": round(e["hz"], 2), "frames": e["n_frames"],
                 "nan_timestamps": e["n_nan_timestamps"], "nan_is_last": e["nan_timestamp_last"],
                 "stated_unit_pupil_xy": e["units"]["data"], "pupil_x_1_99": np.round(e["x_range"], 0).tolist(),
                 "pupil_y_1_99": np.round(e["y_range"], 0).tolist(), "pupil_width_median": round(e["width_median"], 1),
                 "eye_width_median": round(e["eye_width_median"], 1), "frac_nan_pupil": round(e["frac_nan"], 3),
                 "frac_likely_blink": round(e["frac_blink"], 3),
                 "first_frame_after_onset_s": np.round(np.percentile(e["ICwcfg1"]["lag_first_frame"], [0, 100]), 4).tolist()})
eye_tab = pd.DataFrame(rows)
print(eye_tab.to_string(index=False))

Ten sessions have eye tracking, as the Extended Data Fig. 2 legend states. The eye video is sampled
at 60 Hz (timestamps 16.7 ms apart), where the Methods say "Videos of the eye and body were
acquired at 30 Hz"; the paper's code assumes 60 Hz. The pupil position is stated in meters, but its
values (pupil center near 300 to 400, pupil half-width near 20, eye half-width near 150) are camera
pixels, as the code's comments say. No conversion to visual degrees is in the files. One timestamp
per session is NaN, the last one, and is not used. Every IC trial has an eye frame within 17 ms of
its onset, so eye frames cover the trial table; that the eye clock matches the stimulus clock is
consistent with this, not shown by it.

In [ ]:
# The image templates stored in the first session, with the code's names for each frame index.
import h5py, remfile
f = h5py.File(remfile.File(il.session_assets()[0]["url"]), "r")
g = f["stimulus/templates/ICwcfg1_presentations"]
names = ["blank", "X", "TC1", "IC1", "LC1", "TC2", "LC2", "IC2", "IRE1", "IRE2", "TRE1", "TRE2", "XRE1", "XRE2",
         "In BR", "In BL", "In TL", "In TR", "Out BR", "Out BL", "Out TL", "Out TR"]
fig, axs = plt.subplots(4, 6, figsize=(12, 5.5))
for k, ax in enumerate(axs.flat):
    ax.axis("off")
    if k < 22:
        img = g[f"ICwcfg1_presentations{k}"][::10, ::10]
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{k}: {names[k]} ({il.IC_TRIAL_TYPES[k]})", fontsize=7)
plt.tight_layout()
plt.show()
print("template size:", g["ICwcfg1_presentations0"].shape, "; values", g["ICwcfg1_presentations0"][:].min(), "to",
      g["ICwcfg1_presentations0"][:].max())

The templates match the code's names: index 3 and 7 show the two illusory bars (IC1 at 135
degrees, IC2 at 45 degrees), 4 and 6 the recombined LC images, 8 and 9 the IC images with the real
bar outlined (IRE), 10 and 11 the TRE images, 12 and 13 the XRE images, and 14 to 21 single inducer
segments.

In [ ]:
# Recordings deposited twice: compare spike counts, unit counts and trial order between sessions.
sig = []
for s in sessions:
    sig.append((s["subject"], s["n_units_file"], int(s["units"].n_spikes_bins.sum()),
                tuple(s["blocks"]["ICwcfg1"]["codes"][:200]), round(s["blocks"]["ICwcfg1"]["start"][0], 3)))
pairs = [(a[0], b[0]) for a, b in combinations(sig, 2) if a[1] == b[1] or a[2] == b[2] or a[3] == b[3] or a[4] == b[4]]
print("Pairs sharing unit count, spike count, trial order or block start:", pairs if pairs else "none")
print("Trial order identical across sessions:", len({x[3] for x in sig}) == 1)

No two sessions share their unit count, spike count or first block start; every session is a
distinct recording. The trial order of ICwcfg1 is the same in every session (one stimulus script),
which the Methods' "Trial order within each block was randomized" does not contradict.

## results-npx-units-per-session

> The Neuropixels sessions hold 150 +/- 20 V1 RS units and 597 +/- 46 higher visual area RS units
> per session (mean +/- s.e.m., 12 sessions).

In [ ]:
v1 = units_tab["V1_RS"]
hva = units_tab[[f"{a}_RS" for a in il.AREAS[1:]]].sum(axis=1)
print("V1 RS per session:", v1.tolist())
print("HVA RS per session:", hva.tolist())
print("V1 RS: total %d, mean %.1f +/- %.1f (s.e.m.)" % (v1.sum(), v1.mean(), v1.sem()))
print("HVA RS: total %d, mean %.1f +/- %.1f (s.e.m.)" % (hva.sum(), hva.mean(), hva.sem()))
print("Paper: 150 +/- 20 and 597 +/- 46; Methods: 1,804 V1 RS units")
good = [(int(((s["units"].area == "V1") & s["units"].rs & (s["units"].quality == "good")).sum()),
         int(((s["units"].area != "V1") & s["units"].rs & (s["units"].quality == "good")).sum())) for s in sessions]
good = np.array(good)
print("Variant without units labelled noise: V1 %.1f +/- %.1f, HVA %.1f +/- %.1f" %
      (good[:, 0].mean(), stats.sem(good[:, 0]), good[:, 1].mean(), stats.sem(good[:, 1])))

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## methods-npx-ic-encoder-count

> Across the ICwcfg0 and ICwcfg1 blocks, 159 of 1,804 V1 RS units are IC-encoders.

IC-encoders as in the claims file (Kruskal-Wallis across blank, IC1, LC1, LC2, IC2 at p < 0.05,
Tukey-Kramer against blank, AUROC sign; code analyzeStaticICtxi.m), on the IC response
(0 to 400 ms), in each block with that block's IC1 and IC2 images.

In [ ]:
groups = {}
for s in sessions:
    u = s["units"]
    for blk in ["ICwcfg1", "ICwcfg0"]:
        b = s["blocks"][blk]
        g = il.ic_groups(b["R"], b["codes"])
        g.index = u.index
        groups[(s["subject"], blk)] = pd.concat([u[["area", "layer", "rs", "quality"]], g], axis=1)

def v1rs(subject, blk):
    g = groups[(subject, blk)]
    return g[(g.area == "V1") & g.rs]

per = pd.DataFrame([{"subject": s["subject"], "V1_RS": len(v1rs(s["subject"], "ICwcfg1")),
                     "IC_enc_ICwcfg1": int(v1rs(s["subject"], "ICwcfg1").ic_enc.sum()),
                     "IC_enc_ICwcfg0": int(v1rs(s["subject"], "ICwcfg0").ic_enc.sum()),
                     "IC_enc_either": int((v1rs(s["subject"], "ICwcfg1").ic_enc | v1rs(s["subject"], "ICwcfg0").ic_enc).sum())}
                    for s in sessions])
print(per.to_string(index=False))
tot = per.sum(numeric_only=True)
print("\nV1 RS units:", tot.V1_RS)
print("(a) IC-encoder in at least one block:", tot.IC_enc_either)
print("(b) sum of the two blocks' counts:", tot.IC_enc_ICwcfg1 + tot.IC_enc_ICwcfg0)
print("Paper: 159 of 1,804 (tolerance 15.9)")

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed6b-npx-v1-group-counts

> The Neuropixels data hold 87 V1 IC-encoders and 636 V1 segment responders (the sets whose layer
> distribution Extended Data Fig. 6b shows).

Readings of the criterion: (a) RS units, ICwcfg1; (b) all units not classified as noise, ICwcfg1;
(c) RS units, union of ICwcfg1 and ICwcfg0. For (b), the Methods say "All units not classified as
noise were packaged into Neurodata Without Borders files and analyzed in this paper", so on the
paper's own description (b) is every unit in the files; the files also carry a `quality` label
`noise` on some units, so (b) is run both ways: (b1) all units in the files, (b2) without the units
labelled noise.

In [ ]:
def v1all(subject, blk):
    g = groups[(subject, blk)]
    return g[g.area == "V1"]

rows = []
for name, sel in [("(a) RS, ICwcfg1", lambda sub: (v1rs(sub, "ICwcfg1"), None)),
                  ("(b1) all units in the files, ICwcfg1", lambda sub: (v1all(sub, "ICwcfg1"), None)),
                  ("(b2) units not labelled noise, ICwcfg1", lambda sub: (v1all(sub, "ICwcfg1").query("quality != 'noise'"), None)),
                  ("(c) RS, ICwcfg1 or ICwcfg0", lambda sub: (v1rs(sub, "ICwcfg1"), v1rs(sub, "ICwcfg0")))]:
    ic = seg = 0
    for s in sessions:
        g1, g0 = sel(s["subject"])
        if g0 is None:
            ic += int(g1.ic_enc.sum()); seg += int(g1.seg_resp.sum())
        else:
            ic += int((g1.ic_enc | g0.ic_enc).sum()); seg += int((g1.seg_resp | g0.seg_resp).sum())
    rows.append({"reading": name, "IC_encoders": ic, "segment_responders": seg,
                 "IC_within_87+/-8.7": abs(ic - 87) <= 8.7, "seg_within_636+/-63.6": abs(seg - 636) <= 63.6})
print(pd.DataFrame(rows).to_string(index=False))

g = pd.concat([v1all(s["subject"], "ICwcfg1") for s in sessions])
lay = pd.crosstab(g.layer, [g.ic_enc, g.seg_resp])
for name, m in [("IC-encoders", g.ic_enc), ("segment responders", g.seg_resp), ("all V1 units", g.ic_enc | ~g.ic_enc)]:
    print(f"\nLayer proportions, {name} (reading b1), n = {int(m.sum())}:",
          (g[m].layer.value_counts(normalize=True).sort_index().round(3)).to_dict())

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed6cd-npx-ic-encoders-all-areas

> IC-encoders are found in every visual area, in the Neuropixels and in the calcium imaging data.

Neuropixels part only: ICwcfg1, RS units of each area, sessions with at least 50 RS units in the
area. Visually responsive: Kruskal-Wallis across blank, IC1, LC1, LC2, IC2 at p < 0.05.

In [ ]:
rows = []
for s in sessions:
    g = groups[(s["subject"], "ICwcfg1")]
    for a in il.AREAS:
        x = g[(g.area == a) & g.rs]
        if len(x) < 50:
            continue
        vr = x.p_kw_BK < 0.05
        rows.append({"subject": s["subject"], "area": a, "RS": len(x), "IC_enc": int(x.ic_enc.sum()),
                     "vis_resp": int(vr.sum()), "IC_enc_frac_all": x.ic_enc.mean(),
                     "IC_enc_frac_visresp": x.ic_enc[vr].mean() if vr.any() else np.nan})
prop = pd.DataFrame(rows)
summary = prop.groupby("area").agg(sessions=("subject", "size"), IC_encoders=("IC_enc", "sum"), RS_units=("RS", "sum"),
                                   frac_all_mean=("IC_enc_frac_all", "mean"), frac_all_sem=("IC_enc_frac_all", "sem"),
                                   frac_visresp_mean=("IC_enc_frac_visresp", "mean"),
                                   sessions_with_any=("IC_enc", lambda v: int((v > 0).sum()))).reindex(il.AREAS)
print(summary.round(4).to_string())
print("\nAreas with at least one IC-encoder:", [a for a in il.AREAS if summary.loc[a, "IC_encoders"] > 0])

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## fig1e-npx-exclusive-center

> In the Neuropixels data, 18 of 1,804 V1 RS units respond exclusively to the grating patch at the
> center position (the illusory gap region) of the receptive field mapping block.

Per RF trial (four directions, 1 s), the rate 0 to 1000 ms after onset, standard patches only.
Reading (a), the code: signed-rank of each position's 10 trial rates against the unit's mean rate
in the longest spontaneous block; the center is the position of largest mean response and the only
one surviving the code's Holm step. Reading (b), the Methods: rank-sum of each position's trial
rates against the rates in 1-s windows tiled over the longest spontaneous (gray screen) block
(312 windows), Holm over the 9 positions, the center the only significant position (with a mean
above the spontaneous mean, as "responded" implies).

In [ ]:
rfres = {}
for s in sessions:
    u = s["units"]
    m = ((u.area == "V1") & u.rs).values
    d = il.rf_positions(s["rf"]["R"][:, m], s["rf"]["pos"], s["rf"]["inverse"], s["spont_rate"][m], s["spont_windows"][:, m])
    d.index = u.index[m]
    rfres[s["subject"]] = d
rf_tab = pd.DataFrame([{"subject": k, "V1_RS": len(d), "best_center": int((d.best == 0).sum()),
                        "excl_center_a": int(d.excl_a.sum()), "excl_center_b": int(d.excl_b.sum()),
                        "kw_sig": int((d.p_kw_rf < 0.05).sum())} for k, d in rfres.items()])
print(rf_tab.to_string(index=False))
t = rf_tab.sum(numeric_only=True)
print(f"\nV1 RS units {t.V1_RS}; exclusively center-responsive: (a) {t.excl_center_a}, (b) {t.excl_center_b}; paper 18 (tolerance 1.8)")
allrf = pd.concat(rfres.values())
for name, m in [("center best and p_center (a) < 0.05/9", (allrf.best == 0) & (allrf.p_center_a * 9 < 0.05)),
                ("center best and p_center (b) < 0.05/9", (allrf.best == 0) & (allrf.p_center_b * 9 < 0.05)),
                ("(a) and (b) both", allrf.excl_a & allrf.excl_b),
                ("(a), units not labelled noise", allrf.excl_a & np.concatenate([s["units"].quality.values[((s["units"].area == "V1") & s["units"].rs).values] != "noise" for s in sessions]))]:
    print(f"variant, {name}: {int(m.sum())}")

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## fig1f-npx-ic-ire-responsive

> Of the exclusively center-responsive V1 RS units, 66.7% respond substantially to the IC image and
> 77.8% to the real-edge (IRE) image of their preferred IRE orientation, although the IC image has
> no contrast in their receptive field.

Units of fig1e under each of its readings. Preferred IRE orientation: the IRE image with the
strongest mean IC response; the IC image is the one with the illusory bar of that orientation.
Test (a): the code's Kruskal-Wallis across blank, IC and IRE with Tukey-Kramer against blank and
AUROC > 0.5; test (b): rank-sum against blank with AUROC > 0.5. Block (i): ICwcfg1, two
orientations; block (ii): ICwcfg1 and ICwcfg0, four orientations, each image tested in its block.

In [ ]:
def ic_ire_table(sel_by_subject, ic_trials=None):
    """Per selected unit and (test, block reading): IC- and IRE-responsiveness, preferred orientations.
    ic_trials: optional dict subject -> dict block -> boolean mask of trials to keep (fixed gaze)."""
    rows = []
    for s in sessions:
        idx = sel_by_subject.get(s["subject"], [])
        if len(idx) == 0:
            continue
        pos = s["units"].index.get_indexer(idx)
        for blocks in [("ICwcfg1",), ("ICwcfg1", "ICwcfg0")]:
            for test in ["a", "b"]:
                res = {}
                for blk in blocks:
                    b = s["blocks"][blk]
                    keep = np.ones(len(b["codes"]), bool) if ic_trials is None else ic_trials[s["subject"]][blk]
                    R, c = b["R"][keep][:, pos], b["codes"][keep]
                    for k, (icc, irec) in enumerate([(il.IC1, il.IRE1), (il.IC2, il.IRE2)], start=1):
                        icr, irer = il.ic_ire_responsive(R, c, icc, irec, test)
                        res[(blk, k)] = {"ire_mean": R[c == irec].mean(axis=0), "ic_mean": R[c == icc].mean(axis=0),
                                         "ic_resp": icr, "ire_resp": irer}
                keys = list(res)
                ire_means = np.stack([res[k]["ire_mean"] for k in keys], axis=1)
                ic_means = np.stack([res[k]["ic_mean"] for k in keys], axis=1)
                pi = ire_means.argmax(axis=1)
                pc = ic_means.argmax(axis=1)
                for j, unit in enumerate(idx):
                    kk = keys[pi[j]]
                    rows.append({"subject": s["subject"], "unit": unit, "blocks": "+".join(blocks), "test": test,
                                 "pref_ire_ori": il.BAR_ORIENTATION[kk], "pref_ic_ori": il.BAR_ORIENTATION[keys[pc[j]]],
                                 "ic_resp": bool(res[kk]["ic_resp"][j]), "ire_resp": bool(res[kk]["ire_resp"][j])})
    return pd.DataFrame(rows)

def frac_table(t, n_units):
    out = []
    for (blocks, test), x in t.groupby(["blocks", "test"]):
        ire = x[x.ire_resp]
        out.append({"blocks": blocks, "test": test, "units": n_units, "IC_resp_%": 100 * x.ic_resp.mean(),
                    "IRE_resp_%": 100 * x.ire_resp.mean(), "IRE_resp_units": len(ire),
                    "ori_match_%": 100 * (ire.pref_ic_ori == ire.pref_ire_ori).mean() if len(ire) else np.nan})
    return pd.DataFrame(out)

fig1f = {}
for rd in ["a", "b"]:
    sel = {k: d.index[d[f"excl_{rd}"]] for k, d in rfres.items()}
    n = sum(len(v) for v in sel.values())
    t = ic_ire_table(sel)
    fig1f[rd] = t
    ft = frac_table(t, n)
    ft.insert(0, "fig1e_reading", rd)
    print(ft.round(1).to_string(index=False))
print("\nPaper: IC-responsive 66.7% (tolerance 6.67), IRE-responsive 77.8% (7.78); orientation match 78.6% (7.86)")

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## fig1g-npx-orientation-match

> For IRE-responsive exclusively center-responsive V1 RS units, the preferred orientation of the
> illusory bar matches that of the real bar in 78.6% of units.

The `ori_match_%` column above gives the match for every reading; the table below adds the chance
level of each reading (1/number of orientations) and a shuffle of the preferred IC orientations
among the same units (1,000 shuffles, 95th percentile).

In [ ]:
rng = np.random.default_rng(7)
rows = []
for rd, t in fig1f.items():
    for (blocks, test), x in t.groupby(["blocks", "test"]):
        ire = x[x.ire_resp]
        if len(ire) == 0:
            continue
        match = (ire.pref_ic_ori.values == ire.pref_ire_ori.values).mean()
        null = [np.mean(rng.permutation(ire.pref_ic_ori.values) == ire.pref_ire_ori.values) for _ in range(1000)]
        rows.append({"fig1e_reading": rd, "blocks": blocks, "test": test, "IRE_resp_units": len(ire),
                     "match_%": 100 * match, "shuffle_95th_%": 100 * np.percentile(null, 95),
                     "chance_%": 100 / (2 if blocks == "ICwcfg1" else 4)})
print(pd.DataFrame(rows).round(1).to_string(index=False))

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed2b-npx-fixed-gaze

> Limiting the Neuropixels analysis to fixed-gaze trials (pupil within 8 degrees of its modal
> position) leaves 11 exclusively center-responsive V1 RS units, and their responses to ICs and
> orientation match do not change the conclusion.

The files give pupil position in camera pixels only, so reading (a) (8 visual degrees) cannot be
run; reading (b), the code's 20 pixels, is used. Fixed-gaze trial: the largest distance of the
pupil center from its modal position, from the first eye frame after onset to the end of the trial
(1 s for RF trials, 0.4 s for IC trials), is below 20 pixels and no frame is a likely blink (code,
analyzeMouseEye.m). A session is valid when its fixed-gaze standard RF trials cover all 9
positions (code). fig1e, fig1f and fig1g are then repeated on fixed-gaze trials of the valid
sessions.

In [ ]:
fix_rf, fix_ic, valid = {}, {}, []
rows = []
for s in sessions:
    e = s["eye"]
    if e is None:
        continue
    fr = (e["RF"]["maxdist"] < 20) & ~e["RF"]["blink"]
    fi = (e["ICwcfg1"]["maxdist"] < 20) & ~e["ICwcfg1"]["blink"]
    fix_rf[s["subject"]] = fr
    fix_ic[s["subject"]] = {"ICwcfg1": fi, "ICwcfg0": (e["ICwcfg0"]["maxdist"] < 20) & ~e["ICwcfg0"]["blink"]}
    cl = ~s["rf"]["inverse"]
    covered = np.unique(s["rf"]["pos"][fr & cl])
    ok = len(covered) == 9
    if ok:
        valid.append(s["subject"])
    rows.append({"subject": s["subject"], "fixed_RF_trials_%": 100 * fr.mean(), "fixed_standard_RF_trials": int((fr & cl).sum()),
                 "positions_covered": len(covered), "valid": ok, "fixed_ICwcfg1_trials_%": 100 * fi.mean()})
print(pd.DataFrame(rows).round(1).to_string(index=False))
print(f"\nSessions with eye tracking: {len(fix_rf)} (paper 10); valid fixed-gaze RF sessions: {len(valid)} (paper 9)")

rf_fix = {}
for s in sessions:
    if s["subject"] not in valid:
        continue
    u = s["units"]
    m = ((u.area == "V1") & u.rs).values
    d = il.rf_positions(s["rf"]["R"][:, m], s["rf"]["pos"], s["rf"]["inverse"], s["spont_rate"][m],
                        s["spont_windows"][:, m], trials=fix_rf[s["subject"]])
    d.index = u.index[m]
    rf_fix[s["subject"]] = d
for rd in ["a", "b"]:
    n = sum(int(d[f"excl_{rd}"].sum()) for d in rf_fix.values())
    print(f"fixed-gaze exclusively center-responsive V1 RS units, fig1e reading ({rd}): {n} (paper 11, tolerance 1.1)")
rng = np.random.default_rng(11)
rows = []
for rd in ["a", "b"]:
    sel = {k: d.index[d[f"excl_{rd}"]] for k, d in rf_fix.items()}
    n = sum(len(v) for v in sel.values())
    t = ic_ire_table(sel, ic_trials=fix_ic)
    for (blocks, test), x in t.groupby(["blocks", "test"]):
        ire = x[x.ire_resp]
        match = (ire.pref_ic_ori.values == ire.pref_ire_ori.values).mean() if len(ire) else np.nan
        null = [np.mean(rng.permutation(ire.pref_ic_ori.values) == ire.pref_ire_ori.values) for _ in range(1000)] if len(ire) else [np.nan]
        rows.append({"fig1e_reading": rd, "blocks": blocks, "test": test, "units": n, "IC_resp_%": 100 * x.ic_resp.mean(),
                     "IRE_resp_%": 100 * x.ire_resp.mean(), "IRE_resp_units": len(ire), "match_%": 100 * match,
                     "shuffle_95th_%": 100 * np.percentile(null, 95)})
print(pd.DataFrame(rows).round(1).to_string(index=False))

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed2f-npx-rf-match

> Receptive field positions mapped with fixed-gaze trials match those mapped with all trials for
> 81% of the 398 V1 RS units with significant receptive fields.

RF position: the position of largest mean response over standard RF trials. Significant RF:
Kruskal-Wallis across the 9 positions at p < 0.05. Readings: significant with (a) all trials,
(b) fixed-gaze trials, (c) both. Sessions: those with eye tracking whose fixed-gaze trials cover
the 9 positions (the valid sessions of ed2b); the 10 sessions with eye tracking, with the RF
position taken over the positions that have fixed-gaze trials, are shown as a variant.

In [ ]:
def rf_match(subjects):
    rows = []
    for s in sessions:
        if s["subject"] not in subjects:
            continue
        u = s["units"]
        m = ((u.area == "V1") & u.rs).values
        allt = rfres[s["subject"]]
        fx = rf_fix[s["subject"]] if s["subject"] in rf_fix else il.rf_positions(
            s["rf"]["R"][:, m], s["rf"]["pos"], s["rf"]["inverse"], s["spont_rate"][m], s["spont_windows"][:, m],
            trials=fix_rf[s["subject"]])
        rows.append(pd.DataFrame({"subject": s["subject"], "best_all": allt.best.values, "best_fix": fx.best.values,
                                  "sig_all": allt.p_kw_rf.values < 0.05, "sig_fix": fx.p_kw_rf.values < 0.05}))
    return pd.concat(rows)

for name, subs in [("valid sessions", valid), ("all sessions with eye tracking (variant)", list(fix_rf))]:
    t = rf_match(subs)
    print(f"{name}: {t.subject.nunique()} sessions, {len(t)} V1 RS units")
    for rd, m in [("(a) significant with all trials", t.sig_all), ("(b) significant with fixed-gaze trials", t.sig_fix),
                  ("(c) both", t.sig_all & t.sig_fix)]:
        x = t[m]
        print(f"  {rd}: n = {len(x)}, match = {np.mean(x.best_all == x.best_fix):.3f}")
print("Paper: 0.81 (tolerance 0.081) among 398 (39.8) units")

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed2g-npx-pupil-area

> Pupil area during the presentation does not differ across the images of the IC block.

Pupil area z-scored over the session; per trial its mean over the 0 to 0.4 s presentation (end and
maximum as variants); per session and image the mean over trials; Friedman test with sessions as
blocks. Readings: (a) IC1, LC1, LC2, IC2 of ICwcfg1; (b) all 22 images of ICwcfg1.

In [ ]:
eye_sessions = [s for s in sessions if s["eye"] is not None]
names22 = il.IC_TRIAL_TYPES
rows = []
for measure in ["area_mean", "area_end", "area_max"]:
    for rd, imgs in [("(a) IC1, LC1, LC2, IC2", [il.IC1, il.LC1, il.LC2, il.IC2]), ("(b) all 22 images", list(names22))]:
        M = np.array([[np.nanmean(s["eye"]["ICwcfg1"][measure][s["blocks"]["ICwcfg1"]["codes"] == c]) for c in imgs]
                      for s in eye_sessions])
        p = stats.friedmanchisquare(*M.T).pvalue
        rows.append({"measure": measure, "reading": rd, "sessions": len(M), "friedman_p": p,
                     "range_of_image_means_z": np.ptp(M.mean(axis=0))})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Paper: p = 0.7220, Friedman test, 10 sessions")

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed2a-npx-gaze-confined

> Eye gaze is confined to a small region that fits within the illusory gap, with no systematic
> difference in pupil position among IC1, LC1, LC2 and IC2 presentations.

Pupil center during the 0 to 0.4 s presentation of IC1, LC1, LC2 and IC2 in ICwcfg1; per session
the image means of x and of y; a Friedman test with sessions as blocks (the criterion). Per-session
Kruskal-Wallis tests across the four images are a variant. Confinement: the extent, in camera
pixels, of the region where each session's 2D density of pupil positions (2-pixel bins, over the
whole session) is at least 0.01 of its peak, a reading of the criterion's "0.01 isoline".

In [ ]:
imgs = [il.IC1, il.LC1, il.LC2, il.IC2]
rows, kw = [], []
for coord in ["x_mean", "y_mean"]:
    M = np.array([[np.nanmean(s["eye"]["ICwcfg1"][coord][s["blocks"]["ICwcfg1"]["codes"] == c]) for c in imgs]
                  for s in eye_sessions])
    rows.append({"coordinate": coord[0], "friedman_p": stats.friedmanchisquare(*M.T).pvalue,
                 "max_diff_between_image_means_px": np.ptp(M - M.mean(axis=1, keepdims=True), axis=1).max()})
    for s in eye_sessions:
        c = s["blocks"]["ICwcfg1"]["codes"]
        v = s["eye"]["ICwcfg1"][coord]
        kw.append({"subject": s["subject"], "coordinate": coord[0],
                   "kw_p": stats.kruskal(*[v[(c == k) & np.isfinite(v)] for k in imgs]).pvalue})
print(pd.DataFrame(rows).round(4).to_string(index=False))
kw = pd.DataFrame(kw)
print("\nVariant, per-session Kruskal-Wallis across the 4 images: p < 0.05 in", int((kw.kw_p < 0.05).sum()), "of", len(kw),
      "session x coordinate tests; smallest p", kw.kw_p.min().round(4))
ext = []
for s in eye_sessions:
    xy = s["eye"]["density_xy"]
    H, xe, ye = np.histogram2d(xy[:, 0], xy[:, 1], bins=[np.arange(-150, 151, 2)] * 2)
    inside = H >= 0.01 * H.max()
    xs, ys = xe[:-1][inside.any(axis=1)], ye[:-1][inside.any(axis=0)]
    ext.append({"subject": s["subject"], "x_extent_px": xs.max() - xs.min() + 2, "y_extent_px": ys.max() - ys.min() + 2,
                "frac_within_20px_of_mode": np.mean(np.hypot(xy[:, 0], xy[:, 1]) < 20)})
print(pd.DataFrame(ext).round(3).to_string(index=False))

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## fig2d-npx-ccg-hva-input

> V1 IC-encoders (n = 79) are more likely than V1 segment responders (n = 526) to receive putative
> excitatory connections from higher visual areas, by cross-correlogram analysis.

Spike trains of RS units over the ICwcfg1 block in 1-ms bins; the raw CCG for every pair of a V1
IC-encoder or segment responder (RS, ICwcfg1) and an HVA RS unit of the same session; the CCG of
the smoothed trains is subtracted; a connection from the HVA unit when the corrected CCG peak at
lags 1 to 10 ms with the HVA unit leading exceeds 7 times the flank s.d. (50 to 100 ms). Readings
of the criterion: smoothing (a) 25-ms boxcar, (b) 25-ms Gaussian with s.d. 5 ms; flank s.d. from
(i) the raw, (ii) the corrected CCG; measure per V1 unit (x) fraction, (y) number of HVA units
connected to it.

In [ ]:
t0 = time.time()
ccg_rows = []
for s in sessions:
    u = s["units"]
    g = groups[(s["subject"], "ICwcfg1")]
    tg = list(g.index[(g.area == "V1") & g.rs & (g.ic_enc | g.seg_resp)])
    src = list(u.index[(u.area != "V1") & u.rs])
    counts = il.ccg_counts([s["ccg_bins"][i] for i in tg], [s["ccg_bins"][i] for i in src], 124)
    for sm in ["box", "gauss"]:
        for fl in ["raw", "corrected"]:
            con = il.connections(counts, 124, sm, fl)
            for k, i in enumerate(tg):
                ccg_rows.append({"subject": s["subject"], "unit": i, "group": "IC-encoder" if g.loc[i, "ic_enc"] else "segment responder",
                                 "smoothing": sm, "flank": fl, "n_conn": int(con[k].sum()), "frac_conn": con[k].mean(),
                                 "n_hva": len(src)})
ccg = pd.DataFrame(ccg_rows)
print(f"CCGs computed in {time.time() - t0:.0f} s")
x = ccg[(ccg.smoothing == "box") & (ccg.flank == "raw")]
print("Units in the analysis:", x.group.value_counts().to_dict(), "(paper 79 and 526)")
rows = []
for (sm, fl), x in ccg.groupby(["smoothing", "flank"]):
    for meas in ["frac_conn", "n_conn"]:
        a, b = x[x.group == "IC-encoder"][meas], x[x.group == "segment responder"][meas]
        rows.append({"smoothing": sm, "flank_sd_from": fl, "measure": meas, "IC_enc_median": a.median(), "seg_median": b.median(),
                     "IC_enc_mean": a.mean(), "seg_mean": b.mean(), "IC_enc_frac_with_any": (a > 0).mean(),
                     "seg_frac_with_any": (b > 0).mean(), "ranksum_p": stats.ranksums(a, b).pvalue})
ccg_tab = pd.DataFrame(rows)
print(ccg_tab.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
x = ccg[(ccg.smoothing == "box") & (ccg.flank == "raw")]
print("Per session, reading (a)(i), mean fraction of HVA units connected:")
print(x.pivot_table(index="subject", columns="group", values="frac_conn", aggfunc=["mean", "count"]).round(4).to_string())
fig, ax = plt.subplots(figsize=(4, 3))
ax.boxplot([x[x.group == "IC-encoder"].frac_conn, x[x.group == "segment responder"].frac_conn], showfliers=False)
ax.set_xticks([1, 2], ["IC-encoders", "segment responders"])
ax.set_ylabel("fraction of HVA RS units connected")
plt.tight_layout(); plt.show()
print("plotted range (no fliers); all values from", x.frac_conn.min(), "to", round(x.frac_conn.max(), 4))

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## Decoding (Figs. 3a, 3c, 3d; Extended Data Figs. 6e,f and 7)

The decoder of the claims file: ICwcfg1, RS units of one area, sessions with at least 50 RS units
in the area; IC response; classes IC1, LC1, LC2, IC2 balanced to the smallest class by its first
trials; stratified tenfold partition; z-scoring by the training trials; linear SVM, one-vs-one.
Reading (a): C = 1; reading (b): C chosen by an inner threefold cross-validation on the training
trials (the Python counterpart of the code's hyperparameter search). TRE trials are decoded by each
fold model and the fractions averaged. The XRE decoders are trained on XRE1 against XRE2 and probed
with IC1 and IC2. Each (session, area) has its own seed. Reading (b) is run on V1 only: its search
took 30 to 40 s per session and area on this machine, and with reading (a) on every area the
notebook would not have finished within the time of this run (about 25 minutes for both readings
on all areas, estimated from four session-area pairs).

In [ ]:
t0 = time.time()
jobs = []
for i, s in enumerate(sessions):
    u = s["units"]
    for j, a in enumerate(il.AREAS):
        m = ((u.area == a) & u.rs).values
        if m.sum() < 50:
            continue
        b = s["blocks"]["ICwcfg1"]
        jobs.append((s["subject"], a, b["R"][:, m], b["codes"], 1000 * i + j))

def run_decoders(subject, a, R, codes, seed):
    b = {"R": R, "codes": codes}
    m = np.ones(R.shape[1], bool)
    out = []
    for rd in (["a", "b"] if a == "V1" else ["a"]):
        r = il.decode(b["R"][:, m], b["codes"], [il.IC1, il.LC1, il.LC2, il.IC2], [il.TRE1, il.TRE2], seed=seed, reading=rd)
        score, icv, lcv = il.inference_score(r["probe"])
        x = il.decode(b["R"][:, m], b["codes"], [il.XRE1, il.XRE2], [il.IC1, il.IC2], seed=seed, reading=rd)
        out.append({"subject": subject, "area": a, "n_units": int(m.sum()), "reading": rd,
                    "acc": r["acc"], "tre_ic": icv, "tre_lc": lcv, "inference": score, "n_per_class": r["n_per_class"],
                    "xre_acc": x["acc"], "xre_score": (x["probe"][0, 0] + x["probe"][1, 1]) / 2})
    return out

dec = pd.DataFrame([r for rs in Parallel(n_jobs=N_JOBS)(delayed(run_decoders)(*j) for j in jobs) for r in rs])
warnings.filterwarnings("ignore")
print(f"{len(jobs)} session-area decoders, both readings, in {time.time() - t0:.0f} s")
print("Sessions per area:", dec[dec.reading == "a"].groupby("area").size().reindex(il.AREAS).to_dict(),
      "(paper 12, 8, 11, 10, 12, 11)")
print("Trials per class:", sorted(dec.n_per_class.unique()))

## fig3a-npx-v1-decoding

> A linear SVM decodes the four images IC1, LC1, LC2 and IC2 from V1 activity in the Neuropixels
> data with tenfold cross-validation accuracy above the chance of 0.25.

In [ ]:
for rd in ["a", "b"]:
    x = dec[(dec.reading == rd)]
    v = x[x.area == "V1"].acc
    print(f"reading ({rd}) V1: mean accuracy {v.mean():.3f} +/- {v.sem():.3f} (s.e.m.), range {v.min():.3f} to {v.max():.3f}, "
          f"n = {len(v)}, signed-rank against 0.25 p = {il.signrank_p(v - 0.25):.4g}; sessions above 0.25: {(v > 0.25).sum()}")
    if rd == "a":
        print("   other areas:", {a: (round(x[x.area == a].acc.mean(), 3), round(il.signrank_p(x[x.area == a].acc - 0.25), 4))
                                 for a in il.AREAS[1:]})
print(dec[(dec.reading == "a") & (dec.area == "V1")][["subject", "n_units", "acc"]].round(3).to_string(index=False))

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## fig3c-npx-v1-inference

> In Neuropixels V1, TRE trials are decoded as the matching IC image more often than as the
> matching LC image (0.51 against 0.22), so IC inference is represented in V1.

In [ ]:
for rd in ["a", "b"]:
    x = dec[(dec.reading == rd) & (dec.area == "V1")]
    print(f"reading ({rd}): P(TRE -> IC) {x.tre_ic.mean():.3f} +/- {x.tre_ic.sem():.3f}, P(TRE -> LC) {x.tre_lc.mean():.3f} +/- "
          f"{x.tre_lc.sem():.3f}; IC > LC in {(x.tre_ic > x.tre_lc).sum()} of {len(x)} sessions; "
          f"signed-rank IC vs LC p = {il.signrank_p(x.tre_ic, x.tre_lc):.4g}")
print("Paper: 0.51 (tolerance 0.051) and 0.22 (0.022), P < 0.05")
print(dec[(dec.area == "V1")].pivot_table(index="subject", columns="reading", values=["tre_ic", "tre_lc"]).round(3).to_string())

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## fig3d-npx-areas-inference

> In the Neuropixels data almost every visual area decodes TRE as IC more than as LC, significantly
> in V1, LM, RL, AL and PM and not in AM, with V1, LM and AL the strongest.

In [ ]:
paper_p = {"V1": 0.0132, "LM": 0.0156, "RL": 0.0244, "AL": 0.0039, "PM": 0.0342, "AM": 0.1748}
for rd in ["a"]:
    rows = []
    for a in il.AREAS:
        x = dec[(dec.reading == rd) & (dec.area == a)]
        rows.append({"area": a, "sessions": len(x), "IC": x.tre_ic.mean(), "LC": x.tre_lc.mean(), "inference_mean": x.inference.mean(),
                     "inference_sem": x.inference.sem(), "p": il.signrank_p(x.inference), "paper_p": paper_p[a]})
    t = pd.DataFrame(rows)
    t["sig_IC_above_LC"] = (t.p < 0.05) & (t.inference_mean > 0)
    print(f"reading ({rd})")
    print(t.round(4).to_string(index=False))
    print("  significant areas:", t[t.sig_IC_above_LC].area.tolist(),
          "; three largest mean scores:", t.nlargest(3, "inference_mean").area.tolist(),
          "; three smallest p:", t.nsmallest(3, "p").area.tolist())
x = dec[(dec.reading == "b") & (dec.area == "V1")]
print(f"reading (b), V1 only: inference {x.inference.mean():.3f}, p = {il.signrank_p(x.inference):.4g}")

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed7b-npx-xre-inference

> Decoders trained on the XRE real-bar images classify IC1 and IC2 trials as the XRE image whose
> real bar matches the illusory bar, above chance in all six Neuropixels areas.

XRE inference score (P(IC1 -> XRE1) + P(IC2 -> XRE2))/2, signed-rank against 0.5.

In [ ]:
paper_xp = {"V1": 0.0005, "LM": 0.0078, "RL": 0.0049, "AL": 0.0020, "PM": 0.0005, "AM": 0.0137}
for rd in ["a"]:
    rows = []
    for a in il.AREAS:
        x = dec[(dec.reading == rd) & (dec.area == a)]
        rows.append({"area": a, "sessions": len(x), "xre_acc": x.xre_acc.mean(), "xre_score_mean": x.xre_score.mean(),
                     "xre_score_sem": x.xre_score.sem(), "p_vs_0.5": il.signrank_p(x.xre_score - 0.5), "paper_p": paper_xp[a]})
    t = pd.DataFrame(rows)
    t["sig_above_0.5"] = (t["p_vs_0.5"] < 0.05) & (t.xre_score_mean > 0.5)
    print(f"reading ({rd}): areas significant above 0.5: {int(t['sig_above_0.5'].sum())} of 6")
    print(t.round(4).to_string(index=False))
x = dec[(dec.reading == "b") & (dec.area == "V1")]
print(f"reading (b), V1 only: XRE score {x.xre_score.mean():.3f}, p vs 0.5 = {il.signrank_p(x.xre_score - 0.5):.4g}")

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed7a-npx-no-area-above-v1

> No higher visual area shows a substantially larger inference performance than V1, although
> decoding accuracy and inference scores differ across areas.

Kruskal-Wallis across the six areas of the per-session values, then (a) two-sided signed-rank of
V1 against each HVA over the sessions with both, and (b) Tukey-Kramer comparisons after the
Kruskal-Wallis. A claim of absence: reproduced when no HVA is significantly larger than V1.

In [ ]:
def tukey_vs_v1(vals_by_area):
    # Tukey-Kramer on mean ranks after Kruskal-Wallis (MATLAB multcompare default), HVA minus V1
    allv = np.concatenate([vals_by_area[a] for a in il.AREAS])
    lab = np.concatenate([[a] * len(vals_by_area[a]) for a in il.AREAS])
    r = stats.rankdata(allv)
    N = len(allv)
    _, c = np.unique(allv, return_counts=True)
    var = N * (N + 1) / 12 * (1 - np.sum(c ** 3 - c) / (N ** 3 - N))
    qc = il.tukey_qcrit(6)
    out = {}
    for a in il.AREAS[1:]:
        d = r[lab == a].mean() - r[lab == "V1"].mean()
        se = np.sqrt(var * (1 / np.sum(lab == a) + 1 / np.sum(lab == "V1")) / 2)
        out[a] = (round(d, 1), bool(abs(d) / se > qc))
    return out

for rd in ["a"]:
    x = dec[dec.reading == rd]
    print(f"reading ({rd})")
    for meas, paper in [("acc", 0.0019), ("inference", 0.0249), ("xre_acc", 0.0005), ("xre_score", 0.0022)]:
        vals = {a: x[x.area == a][meas].values for a in il.AREAS}
        kwp = stats.kruskal(*vals.values()).pvalue
        piv = x.pivot(index="subject", columns="area", values=meas)
        pair = {}
        for a in il.AREAS[1:]:
            both = piv[["V1", a]].dropna()
            pair[a] = (round(both[a].mean() - both["V1"].mean(), 3), round(il.signrank_p(both[a], both["V1"]), 4))
        print(f"  {meas}: Kruskal-Wallis p = {kwp:.4f} (paper {paper}); means", {a: round(v.mean(), 3) for a, v in vals.items()})
        print(f"     (a) HVA minus V1 and signed-rank p:", pair)
        print(f"     (b) Tukey-Kramer mean-rank difference HVA minus V1, significant:", tukey_vs_v1(vals))

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## ed6ef-npx-ic-not-greater

> In the Neuropixels data, IC images do not evoke larger population average responses than LC
> images in any area, averaged over all units or over visually responsive units.

Sessions with at least 50 RS units in the area; ICwcfg1; per session the IC response averaged over
(e) all RS units, (f) visually responsive RS units (Kruskal-Wallis across blank, IC1, LC1, LC2, IC2
at p < 0.05), on IC trials (IC1 and IC2) and on LC trials (LC1 and LC2); one-tailed signed-rank
test of IC > LC across sessions.

In [ ]:
paper_e = {"V1": 0.9614, "LM": 0.9805, "RL": 0.7676, "AL": 0.7217, "PM": 0.9895, "AM": 0.8398}
paper_f = {"V1": 0.9788, "LM": 0.9805, "RL": 0.7114, "AL": 0.7842, "PM": 0.9866, "AM": 0.6499}
rows = []
for s in sessions:
    u = s["units"]
    b = s["blocks"]["ICwcfg1"]
    g = groups[(s["subject"], "ICwcfg1")]
    for a in il.AREAS:
        m = ((u.area == a) & u.rs).values
        if m.sum() < 50:
            continue
        R = b["R"][:, m]
        icr = R[np.isin(b["codes"], [il.IC1, il.IC2])].mean(axis=0)
        lcr = R[np.isin(b["codes"], [il.LC1, il.LC2])].mean(axis=0)
        vr = (g.p_kw_BK.values[m] < 0.05)
        rows.append({"subject": s["subject"], "area": a, "IC_all": icr.mean(), "LC_all": lcr.mean(),
                     "IC_vr": icr[vr].mean(), "LC_vr": lcr[vr].mean(), "n_vr": int(vr.sum())})
pop = pd.DataFrame(rows)
out = []
for a in il.AREAS:
    x = pop[pop.area == a]
    out.append({"area": a, "sessions": len(x),
                "e_IC_minus_LC_Hz": (x.IC_all - x.LC_all).mean(), "e_p_IC>LC": il.signrank_p(x.IC_all, x.LC_all, "greater"), "e_paper": paper_e[a],
                "f_IC_minus_LC_Hz": (x.IC_vr - x.LC_vr).mean(), "f_p_IC>LC": il.signrank_p(x.IC_vr, x.LC_vr, "greater"), "f_paper": paper_f[a],
                "f_units_mean": x.n_vr.mean()})
pop_tab = pd.DataFrame(out)
print(pop_tab.round(4).to_string(index=False))
print("Areas with p < 0.05 in (e):", pop_tab[pop_tab["e_p_IC>LC"] < 0.05].area.tolist(), "; in (f):", pop_tab[pop_tab["f_p_IC>LC"] < 0.05].area.tolist())

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## supp3-decoder-types

> Decoders of other kinds (quadratic and radial basis function SVMs, a fully connected neural
> network) give results highly consistent with the linear SVM.

The network of the Methods (two hidden layers of 100 units, softmax output, Adam, cross-entropy,
50,000 epochs, tenfold) is timed on one fold of one session, and its full cost estimated. The
quadratic and RBF SVMs were not run: the linear SVM alone took 15 to 20 s per session and area,
and the time of this run did not allow them.

In [ ]:
from sklearn.neural_network import MLPClassifier
s = sessions[0]
b = s["blocks"]["ICwcfg1"]
m = ((s["units"].area == "V1") & s["units"].rs).values
sel = np.isin(b["codes"], [il.IC1, il.LC1, il.LC2, il.IC2])
X = stats.zscore(b["R"][sel][:, m], axis=0)
X[~np.isfinite(X)] = 0
y = b["codes"][sel]
n_ep = 300
net = MLPClassifier(hidden_layer_sizes=(100, 100), solver="adam", batch_size=len(y), max_iter=n_ep, tol=0,
                    n_iter_no_change=10 ** 9, alpha=0.0, random_state=0)
t0 = time.time()
net.fit(X[: int(0.9 * len(y))], y[: int(0.9 * len(y))])
per_epoch = (time.time() - t0) / n_ep
n_fits = 10 * len([j for j in jobs if j[1] in ("V1", "LM", "AL")])
print(f"{per_epoch * 1000:.2f} ms per full-batch epoch on {m.sum()} units; one fit of 50,000 epochs ~ {per_epoch * 5e4 / 60:.1f} min; "
      f"{n_fits} fits (10 folds x {n_fits // 10} session-areas of V1, LM, AL) ~ {per_epoch * 5e4 * n_fits / 3600:.1f} h on one core")

**Result: blocked (too_large).** The author's run reached its time limit before this notebook was executed; it ran to completion afterwards, as the entry check, and its outputs were not reviewed, so no outcome is drawn from the cells above (attempt record, notes).

## Claims on the 2p imaging, 2p mesoscope and 2p holography data

These 19 claims (fig1ik, fig2b, fig2c, ed4c, ed5cd, fig3e (two), ed7b-2p, ed5e, ed4hi, ed3, fig4a,
fig4b, fig5c, fig5, fig5d, fig5e, ed9b, ed10, fig6d, fig6e) rest on recordings that are not in this
dandiset; the paper names no public archive for them (its Data availability lists only this
dandiset for the Neuropixels data and "additional postprocessed data" in a Code Ocean capsule,
10.24433/CO.6659254.v2, which does not say what it holds and could not be reached from this run).
They are blocked with `data_missing`.

In [ ]:
print("Files in the pinned version:", len(assets), "; kinds:", assets.kind.value_counts().to_dict())
print("Any file name mentioning ophys, imaging or 2p:", any(k in p for p in assets.path for k in ["ophys", "imaging", "2p"]))
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| 18 Neuropixels claims | Blocked: too_large (time limit of the author's run; the notebook ran afterwards, outputs not reviewed) |
| supp3-decoder-types | Blocked: too_large |
| 21 claims on 2p imaging, mesoscope and holography | Blocked: data_missing |

How the analysis differs from the paper's: the decoders use scikit-learn's libsvm SVM in place of
MATLAB's fitcecoc, with C = 1 (reading a) and, on V1 only, C chosen by an inner cross-validation
over 10^-3 to 1 (reading b); the paper's code searched hyperparameters with MATLAB's Bayesian
optimisation. The fixed-gaze criterion uses the code's 20 camera pixels, as the files give no
conversion to visual degrees.

What was found about the dataset (attempt record, findings): the files hold units labelled noise
although the Methods say none were packaged; the Methods list 14 mice where the files and the
Results have 12; the pupil position is labelled in meters but is in camera pixels; the eye video is
60 Hz where the Methods say 30 Hz; the IC block tables carry image indices without names; and the
2p data are not in the dandiset.